In [1]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
df=pd.read_csv("all_kindle_review.csv")
df.head(5)

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [3]:
df=df[['rating','reviewText']].dropna()
df.isna().sum()

rating        0
reviewText    0
dtype: int64

In [4]:
df=df.sample(5000)

In [5]:
df.head(5)

,rating,reviewText
844,4,"lose the war,lose your boat lose your girl(may..."
5072,5,"Maybe, maybe not . Mail Order Husband is a s..."
2871,3,"cute premise, two hot students that are hot fo..."
10365,3,I didn't realize when I picked this one up tha...
5563,4,At first I didn't like these super zombies and...


In [6]:
df['rating'] = (df['rating'] > 3).astype(int)

In [7]:
df['rating'].value_counts()

rating
0    2510
1    2490
Name: count, dtype: int64

In [8]:
train_x,test_x,train_y,test_y=train_test_split(df['reviewText'],df['rating'],random_state=42,test_size=0.2,stratify=df['rating'])

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [11]:
train_encodings = tokenizer(
    list(train_x),
    truncation=True,
    padding=True,
    max_length=64
)

test_encodings = tokenizer(
    list(test_x),
    truncation=True,
    padding=True,
    max_length=64
)

In [12]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [13]:
train_dataset = ReviewDataset(train_encodings, train_y)
test_dataset = ReviewDataset(test_encodings, test_y)

In [14]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    acc = accuracy_score(labels, preds)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [16]:
import accelerate

In [17]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    do_eval=False
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [19]:
trainer.train()

  0%|          | 0/250 [00:00<?, ?it/s]

{'train_runtime': 955.1292, 'train_samples_per_second': 4.188, 'train_steps_per_second': 0.262, 'train_loss': 0.4677071838378906, 'epoch': 1.0}


TrainOutput(global_step=250, training_loss=0.4677071838378906, metrics={'train_runtime': 955.1292, 'train_samples_per_second': 4.188, 'train_steps_per_second': 0.262, 'total_flos': 131556708864000.0, 'train_loss': 0.4677071838378906, 'epoch': 1.0})

In [20]:
results = trainer.evaluate()
results

  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 0.4076444208621979,
 'eval_accuracy': 0.823,
 'eval_f1': 0.823000531000531,
 'eval_precision': 0.8230158760635042,
 'eval_recall': 0.823,
 'eval_runtime': 37.8483,
 'eval_samples_per_second': 26.421,
 'eval_steps_per_second': 3.303,
 'epoch': 1.0}

In [21]:
preds = trainer.predict(test_dataset)

y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

  0%|          | 0/125 [00:00<?, ?it/s]

In [22]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.82      0.82       502
           1       0.82      0.83      0.82       498

    accuracy                           0.82      1000
   macro avg       0.82      0.82      0.82      1000
weighted avg       0.82      0.82      0.82      1000

